In [5]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.2 MB/s eta 0:00:00:00:0100:01


In [1]:
import os

os.chdir("C:/Users/pozoy/Desktop/MISIS/final")
print(os.getcwd())
print(os.path.exists("data/ready_dataset.csv"))

C:\Users\pozoy\Desktop\MISIS\final
True


In [2]:
import os
print(os.getcwd())
print(os.path.exists("data/ready_dataset.csv"))

C:\Users\pozoy\Desktop\MISIS\final
True


In [3]:
import os
import json
import random
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from tqdm.auto import tqdm

c:\Users\pozoy\Desktop\MISIS\final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
@dataclass
class Config:
    data_path: str = "data/ready_dataset.csv"
    text_column: str = "combined_text"
    label_column: str = "label"

    model_name: str = "DeepPavlov/rubert-base-cased"
    output_dir: str = "models/rubert"

    max_length: int = 256
    batch_size: int = 8
    epochs: int = 4
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    grad_clip: float = 1.0

    test_size: float = 0.1
    val_size: float = 0.1
    random_state: int = 42
    num_workers: int = 0

    device: str = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def load_dataset(config: Config):
    df = pd.read_csv(config.data_path)

    required_columns = [config.text_column, config.label_column]
    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found. Available columns: {list(df.columns)}")

    df = df[required_columns].copy()
    df = df.dropna()
    df[config.text_column] = df[config.text_column].astype(str).str.strip()
    df = df[df[config.text_column] != ""]

    df[config.label_column] = pd.to_numeric(df[config.label_column], errors="coerce")
    df = df.dropna(subset=[config.label_column])
    df[config.label_column] = df[config.label_column].astype(int)

    labels = sorted(df[config.label_column].unique().tolist())
    if labels != [0, 1]:
        raise ValueError(f"Expected labels [0, 1], got {labels}")

    train_val_df, test_df = train_test_split(
        df,
        test_size=config.test_size,
        random_state=config.random_state,
        stratify=df[config.label_column]
    )

    relative_val_size = config.val_size / (1 - config.test_size)

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=relative_val_size,
        random_state=config.random_state,
        stratify=train_val_df[config.label_column]
    )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True)
    )


class NewsDataset(Dataset):
    def __init__(self, df, tokenizer, text_column, label_column, max_length):
        self.texts = df[text_column].tolist()
        self.labels = df[label_column].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoded = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }


def create_dataloaders(train_df, val_df, test_df, tokenizer, config: Config):
    train_dataset = NewsDataset(
        train_df, tokenizer, config.text_column, config.label_column, config.max_length
    )
    val_dataset = NewsDataset(
        val_df, tokenizer, config.text_column, config.label_column, config.max_length
    )
    test_dataset = NewsDataset(
        test_df, tokenizer, config.text_column, config.label_column, config.max_length
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers
    )

    return train_loader, val_loader, test_loader


def calculate_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, pos_label=1),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    }


def train_epoch(model, dataloader, optimizer, scheduler, device, grad_clip):
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(dataloader, desc="Train", leave=False)

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(len(dataloader), 1)


@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    y_true = []
    y_pred = []

    progress_bar = tqdm(dataloader, desc="Eval", leave=False)

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        total_loss += loss.item()
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())

    metrics = calculate_metrics(y_true, y_pred)
    metrics["loss"] = total_loss / max(len(dataloader), 1)
    metrics["y_true"] = y_true
    metrics["y_pred"] = y_pred
    return metrics


def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)


def save_training_artifacts(config: Config, history, best_epoch, best_val_metrics, test_metrics):
    ensure_dir(config.output_dir)

    results = {
        "rubert": {
            "val_acc": best_val_metrics["accuracy"],
            "val_f1": best_val_metrics["f1"],
            "test_acc": test_metrics["accuracy"],
            "test_f1": test_metrics["f1"],
            "test_precision": test_metrics["precision"],
            "test_recall": test_metrics["recall"],
            "best_epoch": best_epoch,
            "best_params": {
                "model_name": config.model_name,
                "max_length": config.max_length,
                "batch_size": config.batch_size,
                "epochs": config.epochs,
                "learning_rate": config.learning_rate,
                "weight_decay": config.weight_decay
            }
        }
    }

    save_json(os.path.join(config.output_dir, "config.json"), asdict(config))
    save_json(os.path.join(config.output_dir, "history.json"), history)
    save_json(os.path.join(config.output_dir, "metrics.json"), results)

    predictions_df = pd.DataFrame({
        "y_true": test_metrics["y_true"],
        "y_pred": test_metrics["y_pred"]
    })
    predictions_df.to_csv(
        os.path.join(config.output_dir, "test_predictions.csv"),
        index=False,
        encoding="utf-8"
    )


def train_rubert(config: Config):
    set_seed(config.random_state)
    ensure_dir(config.output_dir)

    print("=== Обучение RuBERT ===")
    print(f"Устройство: {config.device}")
    print(f"Файл данных: {config.data_path}")
    print(f"Текстовая колонка: {config.text_column}")
    print(f"Колонка меток: {config.label_column}")
    print("Разметка классов: 0 = fake, 1 = real")

    train_df, val_df, test_df = load_dataset(config)

    print(f"Train size: {len(train_df)}")
    print(f"Val size: {len(val_df)}")
    print(f"Test size: {len(test_df)}")

    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        config.model_name,
        num_labels=2
    ).to(config.device)

    train_loader, val_loader, test_loader = create_dataloaders(
        train_df, val_df, test_df, tokenizer, config
    )

    total_steps = len(train_loader) * config.epochs
    warmup_steps = int(total_steps * config.warmup_ratio)

    optimizer = AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    history = []
    best_val_f1 = -1.0
    best_epoch = -1
    best_val_metrics = None

    for epoch in range(1, config.epochs + 1):
        print(f"\nEpoch {epoch}/{config.epochs}")

        train_loss = train_epoch(
            model, train_loader, optimizer, scheduler, config.device, config.grad_clip
        )

        val_metrics = evaluate(model, val_loader, config.device)

        epoch_result = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_f1": val_metrics["f1"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"]
        }
        history.append(epoch_result)

        print(
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_f1={val_metrics['f1']:.4f}"
        )

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_epoch = epoch
            best_val_metrics = val_metrics

            best_model_dir = os.path.join(config.output_dir, "best_model")
            ensure_dir(best_model_dir)
            model.save_pretrained(best_model_dir)
            tokenizer.save_pretrained(best_model_dir)

            save_json(
                os.path.join(config.output_dir, "best_val_metrics.json"),
                {
                    "epoch": epoch,
                    "val_loss": val_metrics["loss"],
                    "val_acc": val_metrics["accuracy"],
                    "val_f1": val_metrics["f1"],
                    "val_precision": val_metrics["precision"],
                    "val_recall": val_metrics["recall"]
                }
            )

    print(f"\nЛучшая эпоха: {best_epoch}, best_val_f1={best_val_f1:.4f}")

    best_model_dir = os.path.join(config.output_dir, "best_model")
    best_model = AutoModelForSequenceClassification.from_pretrained(best_model_dir).to(config.device)

    test_metrics = evaluate(best_model, test_loader, config.device)

    print("\n=== Test metrics ===")
    print(f"test_loss: {test_metrics['loss']:.4f}")
    print(f"test_acc: {test_metrics['accuracy']:.4f}")
    print(f"test_f1: {test_metrics['f1']:.4f}")
    print(f"test_precision: {test_metrics['precision']:.4f}")
    print(f"test_recall: {test_metrics['recall']:.4f}")

    print("\nClassification report:")
    print(classification_report(test_metrics["y_true"], test_metrics["y_pred"], digits=4))

    save_training_artifacts(config, history, best_epoch, best_val_metrics, test_metrics)

    return {
        "config": asdict(config),
        "history": history,
        "best_epoch": best_epoch,
        "best_val_metrics": best_val_metrics,
        "test_metrics": {
            "accuracy": test_metrics["accuracy"],
            "f1": test_metrics["f1"],
            "precision": test_metrics["precision"],
            "recall": test_metrics["recall"]
        }
    }


def load_inference_artifacts(model_dir="models/rubert/best_model", device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()

    return tokenizer, model, device


@torch.no_grad()
def predict_text(text, tokenizer, model, device, max_length=256):
    encoded = tokenizer(
        str(text),
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    pred = int(np.argmax(probs))

    return {
        "label": pred,
        "label_name": "real" if pred == 1 else "fake",
        "fake_proba": float(probs[0]),
        "real_proba": float(probs[1])
    }


@torch.no_grad()
def predict_texts(texts, tokenizer, model, device, max_length=256):
    results = []
    for text in texts:
        results.append(predict_text(text, tokenizer, model, device, max_length))
    return results


if __name__ == "__main__":
    config = Config()
    train_rubert(config)


=== Обучение RuBERT ===
Устройство: cpu
Файл данных: data/ready_dataset.csv
Текстовая колонка: combined_text
Колонка меток: label
Разметка классов: 0 = fake, 1 = real
Train size: 3526
Val size: 441
Test size: 441


c:\Users\pozoy\Desktop\MISIS\final\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pozoy\.cache\huggingface\hub\models--DeepPavlov--rubert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Dee


Epoch 1/4


Train:   5%|▍         | 22/441 [02:50<53:28,  7.66s/it, loss=0.6991] 